# Example: analyze one session with `navigraph_analysis`

This notebook is intentionally *thin* — it only calls the library. All of the
real logic lives in the `navigraph_analysis` package, and every path comes from
`config/project.yaml`. See `MIGRATION.md` for where each analysis came from.

In [ ]:
from navigraph_analysis import ProjectConfig, Session
from navigraph_analysis.pipelines.session_pipeline import run_session

cfg = ProjectConfig.load()          # config/project.local.yaml or config/project.yaml
session = Session.parse('NPC1', '21_08_2025', cfg)
session, session.output_dir

## Run the full per-session pipeline
Builds the canonical tables (summary/traversal/node/node_edge/reward) then the
behavioral products (turns, decision points, change-of-mind, state features,
quality). Add `neural=True` to attach the spike layer.

In [ ]:
products = run_session(session, cfg, neural=False)
{name: len(df) for name, df in products.tables.items()}

## Inspect a couple of products

In [ ]:
products['turn_df']['turn_type'].value_counts()

In [ ]:
products['direct_opportunities'].head()   # commitment-apex rows (change-of-mind)

## Per-trial context figure and behavioral-state features

In [ ]:
import navigraph_analysis.viz as V
from navigraph_analysis.io.graph import load_maze_graph

G, _ = load_maze_graph(cfg.graph_pkl)
per_trial = V.per_trial_reward(products['reward_df'])
fig = V.trial_context_figure(5, products['summary_df'], products['node_df'], per_trial, G)
fig

In [ ]:
# Design matrix for HMM state inference (fit an HMM downstream; needs [neural] extra)
from navigraph_analysis.behavior.states import build_design_matrix
X, cols = build_design_matrix(products['state_features'])
X.shape, cols

## Cross-session learning curves
Run once all sessions are processed (or via `nga cross`).

In [ ]:
from navigraph_analysis.pipelines.cross_session import build_learning_curves
build_learning_curves(cfg, subject='NPC1')